In [1]:
from skimage import data

import napari

viewer = napari.Viewer()
new_layer = viewer.add_image(data.astronaut(), rgb=True)

In [2]:
import numpy as np
import pandas as pd
import napari
import tifffile
import cv2
import os
import matplotlib.pyplot as plt

In [3]:
cell_type_colors = pd.read_pickle(r'..\results\cell_type_annotation\v4\cell_type_colors.pkl')

In [4]:
cell_type_colors

{'B-cell': '#1f77b4',
 'CD4 T-cell': '#ff7f0e',
 'CD8 T-cell': '#279e68',
 'Dendritic cell': '#d62728',
 'Ductal Epithelial': '#aa40fc',
 'Endothelial cell': '#8c564b',
 'Macrophage': '#e377c2',
 'Mast cell': '#b5bd61',
 'Mucous Acini': '#17becf',
 'Pericytes': '#aec7e8',
 'Plasma cell': '#ffbb78',
 'Seromucous Acini': '#98df8a',
 'Stromal': '#ff9896'}

In [5]:
stromal_type_colors = pd.read_pickle(r'..\results\stromal_subtype_colors.pkl')

In [6]:
stromal_type_colors

{'0': '#1f77b4',
 '1': '#ff7f0e',
 '2': '#279e68',
 '3': '#d62728',
 '4': '#aa40fc',
 '5': '#8c564b',
 '6': '#e377c2',
 '7': '#b5bd61',
 '8': '#17becf',
 '9': '#aec7e8',
 '10': '#ffbb78',
 '11': '#98df8a'}

In [7]:
stromal_subtype = pd.read_csv(r'..\results\stromal_adata_obs.csv', index_col=0)

In [8]:
stromal_subtype

,sample,condition,experiment,CosMx_cluster,cell_types,leiden,neighborhood_cluster
7202_c_aaegfoij-1,7202_c,SSA-,Xenium,2,Stromal,4,3
7202_c_aainlhfo-1,7202_c,SSA-,Xenium,6,Stromal,4,1
7202_c_aajhccbo-1,7202_c,SSA-,Xenium,5,Stromal,2,0
7202_c_aamanhcc-1,7202_c,SSA-,Xenium,7,Stromal,7,4
7202_c_abbphkig-1,7202_c,SSA-,Xenium,7,Stromal,7,1
...,...,...,...,...,...,...,...
rerun2_c_1_30_162,174_2,SSA-,CosMx,6,Stromal,2,3
rerun2_c_1_32_50,174_2,SSA-,CosMx,6,Stromal,8,0
rerun2_c_1_32_88,174_2,SSA-,CosMx,6,Stromal,1,0
rerun2_c_1_33_9,174_2,SSA-,CosMx,7,Stromal,7,5


# xenium 84

In [8]:
spatial_84a = pd.read_csv(r'..\results\cell_type_annotation\v4\xenium_84a_spatial_annotated.csv')

In [9]:
hne_84 = tifffile.imread(os.path.join(r'..\..\20240718_xenium\catalyst_release_AC_EU_Jul18\HnE\0029745_84_a.ome.tif'))

In [10]:
alignment = pd.read_csv(r'..\..\20240718_xenium\catalyst_release_AC_EU_Jul18\HnE\0029745_84_a_alignment_files\matrix.csv', header=None)
alignment = alignment.to_numpy().reshape(3,3)

In [11]:
affine_matrix = alignment[:2]
image_shape = (hne_84.shape[1], hne_84.shape[0])
aligned_hne_84 = cv2.warpAffine(hne_84, affine_matrix, image_shape)

In [12]:
# stromal_84a = stromal_subtype[stromal_subtype['sample'] == '84_a'].copy()
# idx = stromal_84a.index.str.split('_').str[-1]

In [13]:
spatial_84a.index = spatial_84a['cell_id'].tolist()
# stromal_spatial_84a = spatial_84a.loc[idx.tolist()].copy()
# stromal_spatial_84a['leiden'] = stromal_84a['leiden'].tolist()

In [10]:
viewer = napari.Viewer()
viewer.add_image(aligned_hne_84, name='hne_84', colormap='gray')

viewer.add_points(spatial_84a[['y_centroid', 'x_centroid']]/(0.34/1.6), name='spatial_84a', face_color='white', size=2)

for i, cell_type in enumerate(list(cell_type_colors.keys())):
    viewer.add_points(spatial_84a[spatial_84a['cell_types'] == cell_type][['y_centroid', 'x_centroid']]/(0.34/1.6), name=cell_type,
                      face_color=cell_type_colors[cell_type], size=20)

# for i, stromal_type in enumerate(list(stromal_type_colors.keys())):
#     viewer.add_points(stromal_spatial_84a[stromal_spatial_84a['leiden'] == int(stromal_type)][['y_centroid', 'x_centroid']]/(0.34/1.6), symbol='star', name=stromal_type,
#                       face_color=stromal_type_colors[stromal_type], edge_color=stromal_type_colors[stromal_type], size=50)

In [16]:
exp_df = pd.read_csv(r'Y:\coskun-lab\Zhou\12_MSG\20240718_xenium\catalyst_release_AC_EU_Jul18\corrected_84a.csv', index_col=0)

In [39]:
sub_df = exp_df[['LUM', 'IGHG1', 'IGHG2', 'LYZ']]
sub_df = sub_df.loc[spatial_84a.index.tolist()]

In [44]:
size=30

viewer = napari.Viewer()

norm = plt.Normalize(vmin=sub_df['LUM'].min(), vmax=sub_df['LUM'].max())
cmap = plt.cm.Reds
face_colors = cmap(norm(sub_df['LUM'].tolist()))
viewer.add_points(spatial_84a[['y_centroid', 'x_centroid']]/(0.34/1.6), name='LUM', face_color=face_colors, size=size)

norm = plt.Normalize(vmin=sub_df['IGHG1'].min(), vmax=sub_df['IGHG1'].max())
cmap = plt.cm.Reds
face_colors = cmap(norm(sub_df['IGHG1'].tolist()))
viewer.add_points(spatial_84a[['y_centroid', 'x_centroid']]/(0.34/1.6), name='IGHG1', face_color=face_colors, size=size)

norm = plt.Normalize(vmin=sub_df['IGHG2'].min(), vmax=sub_df['IGHG2'].max())
cmap = plt.cm.Reds
face_colors = cmap(norm(sub_df['IGHG2'].tolist()))
viewer.add_points(spatial_84a[['y_centroid', 'x_centroid']]/(0.34/1.6), name='IGHG2', face_color=face_colors, size=size)

norm = plt.Normalize(vmin=sub_df['LYZ'].min(), vmax=sub_df['LYZ'].max())
cmap = plt.cm.Reds
face_colors = cmap(norm(sub_df['LYZ'].tolist()))
viewer.add_points(spatial_84a[['y_centroid', 'x_centroid']]/(0.34/1.6), name='LYZ', face_color=face_colors, size=size)

<Points layer 'LYZ' at 0x1a801418fd0>

# xenium 84b

In [9]:
spatial_84b = pd.read_csv(r'..\results\cell_type_annotation\v3\xenium_84b_spatial_annotation.csv')

In [10]:
hne_84b = tifffile.imread(os.path.join(r'..\..\20240718_xenium\catalyst_release_AC_EU_Jul18\HnE\0029745_84_b.ome.tif'))

In [11]:
alignment = pd.read_csv(r'..\..\20240718_xenium\catalyst_release_AC_EU_Jul18\HnE\0029745_84_b_alignment_files\matrix.csv', header=None)
alignment = alignment.to_numpy().reshape(3,3)

In [12]:
affine_matrix = alignment[:2]
image_shape = (hne_84b.shape[1], hne_84b.shape[0])
aligned_hne_84b = cv2.warpAffine(hne_84b, affine_matrix, image_shape)

In [14]:
viewer = napari.Viewer()
viewer.add_image(aligned_hne_84b, name='hne_84', colormap='gray')

viewer.add_points(spatial_84b[['y_centroid', 'x_centroid']]/(0.34/1.6), name='spatial_84b', face_color='white', size=2)

for i, cell_type in enumerate(list(cell_type_colors.keys())):
    viewer.add_points(spatial_84b[spatial_84b['cell_types'] == cell_type][['y_centroid', 'x_centroid']]/(0.34/1.6), name=cell_type,
                      face_color=cell_type_colors[cell_type], edge_color=cell_type_colors[cell_type], size=20)

# xenium 83

In [9]:
spatial_83a = pd.read_csv(r'..\results\cell_type_annotation\v3\xenium_83a_spatial_annotation.csv')

In [10]:
spatial_83a = spatial_83a[spatial_83a['cell_types'] != 'None']
spatial_83a = spatial_83a[spatial_83a['cell_types'] == spatial_83a['cell_types']]

In [11]:
hne_83 = tifffile.imread(os.path.join(r'..\..\20240718_xenium\catalyst_release_AC_EU_Jul18\HnE\0029866_83_a.ome.tif'))

In [12]:
alignment = pd.read_csv(r'..\..\20240718_xenium\catalyst_release_AC_EU_Jul18\HnE\0029866_83_a_alignment_files\matrix.csv', header=None)
alignment = alignment.to_numpy().reshape(3,3)

In [13]:
affine_matrix = alignment[:2]
image_shape = (hne_83.shape[1], hne_83.shape[0])
aligned_hne_83 = cv2.warpAffine(hne_83, affine_matrix, image_shape)

In [14]:
stromal_83a = stromal_subtype[stromal_subtype['sample'] == '83_a'].copy()
idx = stromal_83a.index.str.split('_').str[-1]

In [15]:
spatial_83a.index = spatial_83a['cell_id'].tolist()
stromal_spatial_83a = spatial_83a.loc[idx.tolist()].copy()
stromal_spatial_83a['leiden'] = stromal_83a['leiden'].tolist()

In [16]:
viewer = napari.Viewer()
viewer.add_image(aligned_hne_83, name='hne_83', colormap='gray')

viewer.add_points(spatial_83a[['y_centroid', 'x_centroid']]/(0.34/1.6), name='spatial_84a', face_color='white', size=2)

for i, cell_type in enumerate(list(cell_type_colors.keys())):
    viewer.add_points(spatial_83a[spatial_83a['cell_types'] == cell_type][['y_centroid', 'x_centroid']]/(0.34/1.6), name=cell_type,
                      face_color=cell_type_colors[cell_type], edge_color=cell_type_colors[cell_type], size=20)

for i, stromal_type in enumerate(list(stromal_type_colors.keys())):
    viewer.add_points(stromal_spatial_83a[stromal_spatial_83a['leiden'] == int(stromal_type)][['y_centroid', 'x_centroid']]/(0.34/1.6), symbol='star', name=stromal_type,
                      face_color=stromal_type_colors[stromal_type], edge_color=stromal_type_colors[stromal_type], size=50)

# xenium 83b

In [21]:
spatial_83b = pd.read_csv(r'..\results\cell_type_annotation\v3\xenium_83b_spatial_annotation.csv')

In [22]:
spatial_83b = spatial_83b[spatial_83b['cell_types'] != 'None']
spatial_83b = spatial_83b[spatial_83b['cell_types'] == spatial_83b['cell_types']]

In [23]:
hne_83b = tifffile.imread(os.path.join(r'..\..\20240718_xenium\catalyst_release_AC_EU_Jul18\HnE\0029866_83_b.ome.tif'))

In [24]:
alignment = pd.read_csv(r'..\..\20240718_xenium\catalyst_release_AC_EU_Jul18\HnE\0029866_83_b_alignment_files\matrix.csv', header=None)
alignment = alignment.to_numpy().reshape(3,3)

In [25]:
affine_matrix = alignment[:2]
image_shape = (hne_83b.shape[1], hne_83.shape[0])
aligned_hne_83b = cv2.warpAffine(hne_83b, affine_matrix, image_shape)

In [27]:
viewer = napari.Viewer()
viewer.add_image(aligned_hne_83b, name='hne_83', colormap='gray')

viewer.add_points(spatial_83b[['y_centroid', 'x_centroid']]/(0.34/1.6), name='spatial_83b', face_color='white', size=2)

for i, cell_type in enumerate(list(cell_type_colors.keys())):
    viewer.add_points(spatial_83b[spatial_83b['cell_types'] == cell_type][['y_centroid', 'x_centroid']]/(0.34/1.6), name=cell_type,
                      face_color=cell_type_colors[cell_type], edge_color=cell_type_colors[cell_type], size=20)

## xenium 174

In [4]:
spatial_174c = pd.read_csv(r'..\results\cell_type_annotation\v3\xenium_174c_spatial_annotation.csv')

In [5]:
spatial_174c = spatial_174c[spatial_174c['cell_types'] != 'None']
spatial_174c = spatial_174c[spatial_174c['cell_types'] == spatial_174c['cell_types']]

In [6]:
hne_174 = tifffile.imread(os.path.join(r'..\..\20240718_xenium\catalyst_release_AC_EU_Jul18\HnE\0029866_174_c.ome.tif'))

In [7]:
alignment = pd.read_csv(r'..\..\20240718_xenium\catalyst_release_AC_EU_Jul18\HnE\0029866_174_c_alignment_files\matrix.csv', header=None)
alignment = alignment.to_numpy().reshape(3,3)
aligned_hne_174 = cv2.warpAffine(hne_174, alignment[:2], (hne_174.shape[1], hne_174.shape[0]))

In [8]:
viewer = napari.Viewer()
viewer.add_image(aligned_hne_174, name='hne_174', colormap='gray')

viewer.add_points(spatial_174c[['y_centroid', 'x_centroid']]/(0.34/1.6), name='spatial_174c', face_color='white', size=2)

for i, cell_type in enumerate(list(cell_type_colors.keys())):
    viewer.add_points(spatial_174c[spatial_174c['cell_types'] == cell_type][['y_centroid', 'x_centroid']]/(0.34/1.6), name=cell_type,
                      face_color=cell_type_colors[cell_type], edge_color=cell_type_colors[cell_type], size=20)

# xenium 174d

In [9]:
spatial_174d = pd.read_csv(r'..\results\cell_type_annotation\v3\xenium_174d_spatial_annotation.csv')

In [10]:
spatial_174d = spatial_174d[spatial_174d['cell_types'] != 'None']
spatial_174d = spatial_174d[spatial_174d['cell_types'] == spatial_174d['cell_types']]

In [11]:
hne_174 = tifffile.imread(os.path.join(r'..\..\20240718_xenium\catalyst_release_AC_EU_Jul18\HnE\0029866_174_d.ome.tif'))

In [12]:
alignment = pd.read_csv(r'..\..\20240718_xenium\catalyst_release_AC_EU_Jul18\HnE\0029866_174_d_alignment_files\matrix.csv', header=None)
alignment = alignment.to_numpy().reshape(3,3)
aligned_hne_174 = cv2.warpAffine(hne_174, alignment[:2], (hne_174.shape[1], hne_174.shape[0]))

In [13]:
stromal_174d = stromal_subtype[stromal_subtype['sample'] == '174_d'].copy()
idx = stromal_174d.index.str.split('_').str[-1]

In [14]:
spatial_174d.index = spatial_174d['cell_id'].tolist()
stromal_spatial_174d = spatial_174d.loc[idx.tolist()].copy()
stromal_spatial_174d['leiden'] = stromal_174d['leiden'].tolist()

In [15]:
viewer = napari.Viewer()
viewer.add_image(aligned_hne_174, name='hne_174', colormap='gray')

viewer.add_points(spatial_174d[['y_centroid', 'x_centroid']]/(0.34/1.6), name='spatial_174d', face_color='white', size=2)

for i, cell_type in enumerate(list(cell_type_colors.keys())):
    viewer.add_points(spatial_174d[spatial_174d['cell_types'] == cell_type][['y_centroid', 'x_centroid']]/(0.34/1.6), name=cell_type,
                      face_color=cell_type_colors[cell_type], edge_color=cell_type_colors[cell_type], size=20)

for i, stromal_type in enumerate(list(stromal_type_colors.keys())):
    viewer.add_points(stromal_spatial_174d[stromal_spatial_174d['leiden'] == int(stromal_type)][['y_centroid', 'x_centroid']]/(0.34/1.6), symbol='star', name=stromal_type,
                      face_color=stromal_type_colors[stromal_type], edge_color=stromal_type_colors[stromal_type], size=50)

c:\Users\zfang38\AppData\Local\anaconda3\envs\napari-env2\lib\site-packages\napari\utils\migrations.py:101: FutureWarning: Argument 'edge_color' is deprecated, please use 'border_color' instead. The argument 'edge_color' was deprecated in 0.5.0 and it will be removed in 0.6.0.
  return func(*args, **kwargs)
c:\Users\zfang38\AppData\Local\anaconda3\envs\napari-env2\lib\site-packages\napari\utils\migrations.py:101: FutureWarning: Argument 'edge_color' is deprecated, please use 'border_color' instead. The argument 'edge_color' was deprecated in 0.5.0 and it will be removed in 0.6.0.
  return func(*args, **kwargs)
c:\Users\zfang38\AppData\Local\anaconda3\envs\napari-env2\lib\site-packages\napari\utils\migrations.py:101: FutureWarning: Argument 'edge_color' is deprecated, please use 'border_color' instead. The argument 'edge_color' was deprecated in 0.5.0 and it will be removed in 0.6.0.
  return func(*args, **kwargs)
c:\Users\zfang38\AppData\Local\anaconda3\envs\napari-env2\lib\site-packag

## xenium 7202

In [66]:
spatial_7202c = pd.read_csv(r'..\results\cell_type_annotation\v3\xenium_7202c_spatial_annotation.csv')

In [67]:
spatial_7202c = spatial_7202c[spatial_7202c['cell_types'] != 'None']
spatial_7202c = spatial_7202c[spatial_7202c['cell_types'] == spatial_7202c['cell_types']]

In [68]:
hne_7202 = tifffile.imread(os.path.join(r'..\..\20240718_xenium\catalyst_release_AC_EU_Jul18\HnE\0029745_7202_c.ome.tif'))

In [69]:
alignment = pd.read_csv(r'..\..\20240718_xenium\catalyst_release_AC_EU_Jul18\HnE\0029745_7202_c_alignment_files\matrix.csv', header=None)
alignment = alignment.to_numpy().reshape(3,3)
aligned_hne_7202 = cv2.warpAffine(hne_7202, alignment[:2], (hne_7202.shape[1], hne_7202.shape[0]))

In [70]:
stromal_7202c = stromal_subtype[stromal_subtype['sample'] == '7202_c'].copy()
idx = stromal_7202c.index.str.split('_').str[-1]

In [71]:
spatial_7202c.index = spatial_7202c['cell_id'].tolist()
stromal_spatial_7202c = spatial_7202c.loc[idx.tolist()].copy()
stromal_spatial_7202c['leiden'] = stromal_7202c['leiden'].tolist()

In [73]:
viewer = napari.Viewer()
viewer.add_image(aligned_hne_7202, name='hne_7202', colormap='gray')

viewer.add_points(spatial_7202c[['y_centroid', 'x_centroid']]/(0.34/1.6), name='spatial_7202c', face_color='white', size=2)

for i, cell_type in enumerate(list(cell_type_colors.keys())):
    viewer.add_points(spatial_7202c[spatial_7202c['cell_types'] == cell_type][['y_centroid', 'x_centroid']]/(0.34/1.6), name=cell_type,
                      face_color=cell_type_colors[cell_type], edge_color=cell_type_colors[cell_type], size=20)

for i, stromal_type in enumerate(list(stromal_type_colors.keys())):
    viewer.add_points(stromal_spatial_7202c[stromal_spatial_7202c['leiden'] == int(stromal_type)][['y_centroid', 'x_centroid']]/(0.34/1.6), symbol='star', name=stromal_type,
                      face_color=stromal_type_colors[stromal_type], edge_color=stromal_type_colors[stromal_type], size=50)

RuntimeError: OpenGL got errors (periodic check): GL_OUT_OF_MEMORY

# xenium 7202d

In [45]:
spatial_7202d = pd.read_csv(r'..\results\cell_type_annotation\v3\xenium_7202d_spatial_annotation.csv')

In [46]:
spatial_7202d = spatial_7202d[spatial_7202d['cell_types'] != 'None']
spatial_7202d = spatial_7202d[spatial_7202d['cell_types'] == spatial_7202d['cell_types']]

In [47]:
hne_7202 = tifffile.imread(os.path.join(r'..\..\20240718_xenium\catalyst_release_AC_EU_Jul18\HnE\0029745_7202_d.ome.tif'))

In [48]:
alignment = pd.read_csv(r'..\..\20240718_xenium\catalyst_release_AC_EU_Jul18\HnE\0029745_7202_d_alignment_files\matrix.csv', header=None)
alignment = alignment.to_numpy().reshape(3,3)
aligned_hne_7202 = cv2.warpAffine(hne_7202, alignment[:2], (hne_7202.shape[1], hne_7202.shape[0]))

In [49]:
viewer = napari.Viewer()
viewer.add_image(aligned_hne_7202, name='hne_7202', colormap='gray')

viewer.add_points(spatial_7202d[['y_centroid', 'x_centroid']]/(0.34/1.6), name='spatial_7202c', face_color='white', size=2)

for i, cell_type in enumerate(list(cell_type_colors.keys())):
    viewer.add_points(spatial_7202d[spatial_7202d['cell_types'] == cell_type][['y_centroid', 'x_centroid']]/(0.34/1.6), name=cell_type,
                      face_color=cell_type_colors[cell_type], edge_color=cell_type_colors[cell_type], size=20)

# cosmx 84

In [50]:
cosmx_84_spatial = pd.read_csv(r'..\results\cell_type_annotation\v3\cosmx_84_spatial_annotation.csv')

In [51]:
viewer = napari.Viewer()
viewer.add_points(-cosmx_84_spatial[['CenterY_global_px', 'CenterX_global_px']], name='cosmx_84', face_color='white', size=20)

for i, cell_type in enumerate(cosmx_84_spatial['cell_types'].unique()):
    viewer.add_points(-cosmx_84_spatial[cosmx_84_spatial['cell_types'] == cell_type][['CenterY_global_px', 'CenterX_global_px']], name=cell_type,
                      face_color=cell_type_colors[cell_type], edge_color=cell_type_colors[cell_type], size=50)

# cosmx 83

In [69]:
cosmx_spatial_83 = pd.read_csv(r'..\..\20240615_nanostring\GATech_Ahmet_1st\Flat_File\83005\83005_metadata_file.csv.gz')

In [70]:
cosmx_sub_83 = cosmx_spatial_83[['cell','CenterX_global_px','CenterY_global_px']]

In [71]:
cosmx_sub_83 = cosmx_sub_83.merge(labeled_cells, left_on='cell', right_on='cell_id')
cosmx_sub_83 = cosmx_sub_83.dropna(subset=['cell_types'], axis=0)

In [72]:
viewer = napari.Viewer()
viewer.add_points(-cosmx_sub_83[['CenterY_global_px', 'CenterX_global_px']], name='cosmx_83', face_color='white', size=20)

for i, cell_type in enumerate(cosmx_sub_83['cell_types'].unique()):
    viewer.add_points(-cosmx_sub_83[cosmx_sub_83['cell_types'] == cell_type][['CenterY_global_px', 'CenterX_global_px']], name=cell_type,
                      face_color=color_dict[cell_type], edge_color=color_dict[cell_type], size=50)

# cosmx 174

In [73]:
cosmx_spatial_174 = pd.read_csv(r'..\..\20240615_nanostring\GATech_Ahmet_1st\Flat_File\174003\174003_metadata_file.csv.gz')

In [74]:
cosmx_sub_174 = cosmx_spatial_174[['cell','CenterX_global_px','CenterY_global_px']]

In [75]:
cosmx_sub_174 = cosmx_sub_174.merge(labeled_cells, left_on='cell', right_on='cell_id')
cosmx_sub_174 = cosmx_sub_174.dropna(subset=['cell_types'], axis=0)

In [76]:
viewer = napari.Viewer()
viewer.add_points(-cosmx_sub_174[['CenterY_global_px', 'CenterX_global_px']], name='cosmx_83', face_color='white', size=20)

for i, cell_type in enumerate(cosmx_sub_174['cell_types'].unique()):
    viewer.add_points(-cosmx_sub_174[cosmx_sub_174['cell_types'] == cell_type][['CenterY_global_px', 'CenterX_global_px']], name=cell_type,
                      face_color=color_dict[cell_type], edge_color=color_dict[cell_type], size=50)

# coxmx 7202

In [77]:
cosmx_spatial_7202 = pd.read_csv(r'..\..\20240615_nanostring\GATech_Ahmet_1st\Flat_File\7202002\7202002_metadata_file.csv.gz')

In [78]:
cosmx_sub_7202 = cosmx_spatial_7202[['cell','CenterX_global_px','CenterY_global_px']]

In [79]:
cosmx_sub_7202 = cosmx_sub_7202.merge(labeled_cells, left_on='cell', right_on='cell_id')
cosmx_sub_7202 = cosmx_sub_7202.dropna(subset=['cell_types'], axis=0)

In [80]:
viewer = napari.Viewer()
viewer.add_points(-cosmx_sub_7202[['CenterY_global_px', 'CenterX_global_px']], name='cosmx_83', face_color='white', size=20)

for i, cell_type in enumerate(cosmx_sub_7202['cell_types'].unique()):
    viewer.add_points(-cosmx_sub_7202[cosmx_sub_7202['cell_types'] == cell_type][['CenterY_global_px', 'CenterX_global_px']], name=cell_type,
                      face_color=color_dict[cell_type], edge_color=color_dict[cell_type], size=50)

# temp

In [88]:
import os
import pickle

In [116]:
temp = pd.read_csv(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\17472022\17472022_metadata_file.csv.gz')

In [117]:
viewer = napari.Viewer()
points_layer = viewer.add_points(-temp[['CenterY_global_px', 'CenterX_global_px']], name='temp', face_color='white', size=70)

In [122]:
indices = points_layer.selected_data

In [120]:
fov_7202 = temp['fov'][indices].unique()

In [121]:
fov_7202

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18], dtype=int64)

In [123]:
fov_174 = temp['fov'][indices].unique()
fov_174

array([29, 30, 31, 32, 33, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28],
      dtype=int64)

In [124]:
with open(os.path.join(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\sample_split','7202_2_fov.pkl'), 'wb') as f:
    pickle.dump(fov_7202, f)
with open(os.path.join(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\sample_split','174_2_fov.pkl'), 'wb') as f:
    pickle.dump(fov_174, f)

In [126]:
fov_1741 = pickle.load(open(os.path.join(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\sample_split','174_1_cells.pkl'), 'rb'))
fov_72021 = pickle.load(open(os.path.join(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\sample_split','7202_1_cells.pkl'), 'rb'))
fov_1742 = pickle.load(open(os.path.join(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\sample_split','174_2_fov.pkl'), 'rb'))
fov_72022 = pickle.load(open(os.path.join(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\sample_split','7202_2_fov.pkl'), 'rb'))

In [132]:
# split exprMat_file
file = pd.read_csv(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\17472022\17472022_exprMat_file.csv.gz', compression='gzip')
exprMat_7202 = file[file['fov'].isin(fov_72022)]
exprMat_174 = file[file['fov'].isin(fov_1742)]

exprMat_7202.to_csv(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\7202_2\7202_2_exprMat_file.csv.gz', index=False, compression='gzip')
exprMat_174.to_csv(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\174_2\174_2_exprMat_file.csv.gz', index=False, compression='gzip')

In [133]:
# split fov_positions_file
file = pd.read_csv(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\17472022\17472022_fov_positions_file.csv.gz', compression='gzip')
fov_position_7202 = file[file['FOV'].isin(fov_72022)]
fov_position_174 = file[file['FOV'].isin(fov_1742)]

fov_position_7202.to_csv(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\7202_2\7202_2_fov_positions_file.csv.gz', index=False, compression='gzip')
fov_position_174.to_csv(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\174_2\174_2_fov_positions_file.csv.gz', index=False, compression='gzip')

In [134]:
# split metadata_file
file = pd.read_csv(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\17472022\17472022_metadata_file.csv.gz', compression='gzip')
metadata_7202 = file[file['fov'].isin(fov_72022)]
metadata_174 = file[file['fov'].isin(fov_1742)]

metadata_7202.to_csv(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\7202_2\7202_2_metadata_file.csv.gz', index=False, compression='gzip')
metadata_174.to_csv(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\174_2\174_2_metadata_file.csv.gz', index=False, compression='gzip')

In [135]:
# split tx_file
file = pd.read_csv(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\17472022\17472022_tx_file.csv.gz', compression='gzip')
tx_7202 = file[file['fov'].isin(fov_72022)]
tx_174 = file[file['fov'].isin(fov_1742)]

tx_7202.to_csv(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\7202_2\7202_2_tx_file.csv.gz', index=False, compression='gzip')
tx_174.to_csv(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\174_2\174_2_tx_file.csv.gz', index=False, compression='gzip')

C:\Users\zfang38\AppData\Local\Temp\ipykernel_46436\3123798866.py:2: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  file = pd.read_csv(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\17472022\17472022_tx_file.csv.gz', compression='gzip')


In [136]:
# Split polygons
file = pd.read_csv(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\17472022\17472022-polygons.csv.gz', compression='gzip')
polygons_7202 = file[file['fov'].isin(fov_72022)]
polygons_174 = file[file['fov'].isin(fov_1742)]

polygons_7202.to_csv(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\7202_2\7202_2-polygons.csv.gz', index=False, compression='gzip')
polygons_174.to_csv(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\174_2\174_2-polygons.csv.gz', index=False, compression='gzip')

In [85]:
exprMat = pd.read_csv(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\17472021\17472021_exprMat_file.csv.gz', compression='gzip')
metadata = pd.read_csv(r'..\..\20240827_cosmx\GATech_Ahmet_2nd\174-7202_1\Flat_File\17472021\17472021_metadata_file.csv.gz', compression='gzip')